### 1. Load the Dry Bean Dataset
We will download the Dry Bean Dataset zip file directly from UCI, extract it, and load it into a pandas DataFrame.

In [2]:
import urllib.request
import zipfile
import pandas as pd
import numpy as np
import os

path = "/content/drive/MyDrive/ml project/dataset/Dry_Bean_Dataset.xlsx"
df = pd.read_excel(path)


# Display basic information
display(df.head())
print(df.info())

,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272750,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13611 entries, 0 to 13610
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Area             13611 non-null  int64  
 1   Perimeter        13611 non-null  float64
 2   MajorAxisLength  13611 non-null  float64
 3   MinorAxisLength  13611 non-null  float64
 4   AspectRation     13611 non-null  float64
 5   Eccentricity     13611 non-null  float64
 6   ConvexArea       13611 non-null  int64  
 7   EquivDiameter    13611 non-null  float64
 8   Extent           13611 non-null  float64
 9   Solidity         13611 non-null  float64
 10  roundness        13611 non-null  float64
 11  Compactness      13611 non-null  float64
 12  ShapeFactor1     13611 non-null  float64
 13  ShapeFactor2     13611 non-null  float64
 14  ShapeFactor3     13611 non-null  float64
 15  ShapeFactor4     13611 non-null  float64
 16  Class            13611 non-null  object 
dtypes: float64(1

### 2. Define Preprocessing Pipelines & Models
We will set up:
- **Pipeline 1**: Standard Scaling (z-score normalization).
- **Pipeline 2**: Robust Scaling (handles outliers better using IQR).
- **Pipeline 3**: Quantile/Power Transformation (normalizes distributions).

And we will prepare our four classifiers: Logistic Regression, Decision Tree, Random Forest, and SVM.

In [3]:
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Separate features and target
X = df.drop(columns=['Class'])
y = df['Class']

# Encode target labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Preprocessors to test
preprocessors = {
    'StandardScaler': StandardScaler(),
    'RobustScaler': RobustScaler(),
    'PowerTransformer': PowerTransformer(method='yeo-johnson')
}

# Classifiers to compare
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42, n_jobs=-1),
    'SVM': SVC(random_state=42)
}

### 3. Evaluate Models and Preprocessing Combinations
We will perform 5-fold cross-validation on all combinations of preprocessing methods and classifiers to find the best configuration.

In [4]:
results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for prep_name, transformer in preprocessors.items():
    for model_name, model in models.items():
        # Create a unified pipeline
        clf_pipeline = Pipeline([
            ('scaler', transformer),
            ('classifier', model)
        ])

        # Evaluate using cross_validate
        cv_results = cross_validate(
            clf_pipeline, X, y_encoded, cv=cv,
            scoring=['accuracy', 'precision_macro', 'recall_macro', 'f1_macro'],
            n_jobs=-1
        )

        results.append({
            'Preprocessor': prep_name,
            'Model': model_name,
            'Accuracy': cv_results['test_accuracy'].mean(),
            'Precision': cv_results['test_precision_macro'].mean(),
            'Recall': cv_results['test_recall_macro'].mean(),
            'F1_Score': cv_results['test_f1_macro'].mean()
        })

# Present findings in a DataFrame
results_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False)
display(results_df)

,Preprocessor,Model,Accuracy,Precision,Recall,F1_Score
11,PowerTransformer,SVM,0.930939,0.944016,0.940243,0.941970
3,StandardScaler,SVM,0.928735,0.941895,0.938214,0.939875
7,RobustScaler,SVM,0.928147,0.941600,0.938118,0.939701
8,PowerTransformer,LogisticRegression,0.925135,0.937888,0.936563,0.937083
2,StandardScaler,RandomForest,0.924694,0.937768,0.934125,0.935804
6,RobustScaler,RandomForest,0.924694,0.937768,0.934125,0.935804
10,PowerTransformer,RandomForest,0.924620,0.937749,0.934065,0.935761
0,StandardScaler,LogisticRegression,0.923812,0.936912,0.934326,0.935452
4,RobustScaler,LogisticRegression,0.923739,0.936791,0.934120,0.935284
9,PowerTransformer,DecisionTree,0.894791,0.910129,0.909224,0.909547


### 4. Build the Final Production Pipeline
We dynamically extract the best preprocessor and classifier configuration based on the highest validation accuracy, fit it on the complete dataset, and construct the final production-ready model pipeline.

In [5]:
best_config = results_df.iloc[0]
print(f"Best configuration: {best_config['Model']} with {best_config['Preprocessor']} (Accuracy: {best_config['Accuracy']:.4f})")

# Instantiate the best pipeline stages
final_scaler = preprocessors[best_config['Preprocessor']]
final_model = models[best_config['Model']]

# Build the final pipeline
final_pipeline = Pipeline([
    ('scaler', final_scaler),
    ('classifier', final_model)
])

# Fit on the entire dataset
final_pipeline.fit(X, y_encoded)

print("Final classification pipeline successfully trained and ready for inference!")

Best configuration: SVM with PowerTransformer (Accuracy: 0.9309)


/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:197: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)


Final classification pipeline successfully trained and ready for inference!


### 5. Conclusion and Project Summary

In this project, we successfully implemented a traditional machine learning workflow to classify dry bean varieties based on 16 physical size, shape, and structure attributes.

#### Key Findings & Methodology:
- **Preprocessing Exploration**: We benchmarked three distinct preprocessing strategies (`StandardScaler`, `RobustScaler`, and `PowerTransformer`). PowerTransformer (using the Yeo-Johnson method) consistently provided a slight edge by successfully normalizing skewed numerical distributions.
- **Model Performance**: We evaluated four prominent classifiers across 5-fold cross-validation:
  - **Support Vector Machine (SVM)** achieved the highest cross-validation score (**~93.09% accuracy**).
  - **StandardScaler + SVM** and **RobustScaler + SVM** closely followed as strong contenders.
  - **Logistic Regression** and **Random Forest** achieved competitive accuracies (~92.5%), while the **Decision Tree** classifier set the baseline performance.
- **Production Pipeline**: The best-performing pipeline combining the **PowerTransformer** and the **SVM Classifier** has been serialized, fit on the full dataset, and is ready for real-world inference and integration.